# COCO Caption Classification with Zero-shot and Few-shot Learning

This notebook focuses on Multimodal Classification using the COCO dataset. It evaluates the ability to align visual semantics with complex linguistic context.

## Setup and Configurations

In [ ]:
# System dependencies
!pip install -r ../settings/requirements.txt
!pip install wandb -qU

import sys
import os

# Assuming you cloned the repository and opened this notebook on Colab, 
# the working directory should be the notebooks/ folder inside the repo.
if not os.path.exists('../src') and not os.path.exists('src'):
    print("Cloning repository...")
    # NOTE: Replace with your actual github repository URL.
    !git clone https://github.com/ThaiLearnCoding/Coco_caption_classification.git
    os.chdir('coco_caption_classification/notebooks')

print("Working directory:", os.getcwd())
sys.path.append(os.path.abspath(os.path.join('..')))

import torch
import yaml
import wandb

# Config
config_path = '../settings/config.yaml'
try:
    with open(config_path, 'r') as f:
         config = yaml.safe_load(f)
except FileNotFoundError:
    print("Config file not found, using generic defaults")
    config = {
        "training": {
            "k_shots": [8, 16, 32],
            "device": "cuda" if torch.cuda.is_available() else "cpu",
            "epochs": 20
        },
        "models": {"clip_vit": "ViT-B/32", "clip_rn50": "RN50"}
    }

# Enable or disable training
TRAIN_MODEL = True # Set to False if you just want to run inference

device = config['training']['device'] if 'training' in config else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Dataset Retrieval and Loading

We load the MS COCO multimodal subset from `coco_subset_images/coco_multimodal_subset.json`

In [ ]:
# 1. Install gdown (usually pre-installed in Colab, but upgrade for large files)
!pip install --upgrade gdown

import os
import gdown
import json

# --- CONFIGURATION ---
ZIP_FILE_ID = '1ZZ6NBl5567WZVRTCNspteJ_IiZeRr352' # Paste your ID here
JSON_FILE_ID = '1-WsEVMu4Ubxr-rV6LvgxhMh2Voe3HZwU' # Paste your JSON ID here
ZIP_DEST = 'coco_data.zip'
EXTRACT_DIR = '/content/coco_subset_images'
# ---------------------

# 1. Download the Zip file from Drive
if not os.path.exists(ZIP_DEST):
    print("Downloading 2GB Image Archive...")
    url = f'https://drive.google.com/uc?id={ZIP_FILE_ID}'
    gdown.download(url, ZIP_DEST, quiet=False)

# 2. Extract the Zip
if not os.path.exists(EXTRACT_DIR):
    print("Extracting images... this may take a minute.")
    !unzip -q {ZIP_DEST} -d /content/
    print("Extraction complete.")

# 3. Download the JSON metadata
if not os.path.exists('coco_subset.json'):
    print("Downloading JSON metadata...")
    json_url = f'https://drive.google.com/uc?id={JSON_FILE_ID}'
    gdown.download(json_url, 'coco_subset.json', quiet=False)

# 4. Load the dataset
with open('coco_subset.json', 'r') as f:
    final_dataset = json.load(f)

print(f"✅ Setup Complete. {len(final_dataset)} samples loaded and images ready at {EXTRACT_DIR}")

In [ ]:
from src import data_loader, model_arch, utils

# Inherit the dataset loaded in the Setup cell
try:
    subset_data = final_dataset
    print(f"Loaded {len(subset_data)} samples.")
except NameError:
    print("final_dataset is not defined. Make sure you run the prior dataset downloading cell first.")
    subset_data = []

## Exploratory Data Analysis (EDA)

Here we visualize the class distribution using plotly and show a random 3x3 grid of images along with their ground-truth caption (green) and distractor captions (red).

In [ ]:
import plotly.express as px
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import os
import random

if subset_data:
    # Plotly bar chart
    counts = Counter([item['category'] for item in subset_data])
    df = pd.DataFrame(counts.items(), columns=['Category', 'Count']).sort_values('Count', ascending=False)
    
    fig = px.bar(df, x='Category', y='Count', title='Data Distribution per Category (COCO Subset)',
                 color='Category', template='plotly_white')
    fig.show()

    # 3x3 Grid visualization
    fig_grid, axes = plt.subplots(3, 3, figsize=(15, 15))
    samples = random.sample(subset_data, 9)
    # Reference the global variable for extraction path defined in cell 1 setup 
    image_dir = EXTRACT_DIR if 'EXTRACT_DIR' in locals() else '/content/coco_subset_images'

    for i, ax in enumerate(axes.flat):
        sample = samples[i]
        img_path = os.path.join(image_dir, sample['file_name'])
        
        try:
            img = Image.open(img_path)
            ax.imshow(img)
            ax.axis('off')
            
            # Ground truth in green, 1 distractor in red for visualization (to keep it readable)
            gt = sample['caption']
            distractor = sample['distractors'][0] if sample['distractors'] else "No distractor"
            
            text = f"GT: {gt[:30]}...\nDist: {distractor[:30]}..."
            ax.set_title(text, fontsize=10, color='green', pad=2)
            
            # Highlight distractor part
            ax.text(0.5, -0.1, f"Distractor: {distractor[:30]}...", ha='center', va='top', transform=ax.transAxes, color='red', fontsize=10)
        except Exception as e:
            ax.set_title("Image missing")
            ax.axis('off')

    plt.tight_layout()
    plt.show()

## Few-Shot Data Splitting

We isolate 32 training samples per class and remove them from the evaluation subset. We extract subsets for $k=8$, $k=16$, and $k=32$.

In [ ]:
train_max_data, eval_data = data_loader.create_few_shot_splits(subset_data, n_max_shots=32, seed=42)

print(f"Max train pool size: {len(train_max_data)}")
print(f"Evaluation set size: {len(eval_data)}")

## Model Architecture and Initialization

Load the CLIP variants (`ViT-B/32` and `RN50`). Here we prepare DataLoaders that apply the architecture-specific normalization depending on the chosen model. We then set up the Model with Custom Classification Head (Linear Probing).

In [ ]:
import clip

clip_model_name_vit = config['models']['clip_vit']
clip_model_name_rn50 = config['models']['clip_rn50']

print("Initializing ViT-B/32 Zero-Shot Model")
vit_zs_model = model_arch.CLIPZeroShotClassifier(model_name=clip_model_name_vit, device=device)
preprocess_vit = vit_zs_model.preprocess

print("Initializing RN50 Zero-Shot Model")
rn50_zs_model = model_arch.CLIPZeroShotClassifier(model_name=clip_model_name_rn50, device=device)
preprocess_rn50 = rn50_zs_model.preprocess

print("Initializing linear probe models for fine-tuning")
# Linear Probing head mapping to similarity score with Cross-Entropy Loss
vit_lp_model = model_arch.LinearProbeClassifier(model_name=clip_model_name_vit, device=device, n_captions=5)
rn50_lp_model = model_arch.LinearProbeClassifier(model_name=clip_model_name_rn50, device=device, n_captions=5)

# Size & param counts
print("\nViT Size Info:")
vit_params, vit_train_params, vit_size_mb = utils.calculate_model_size_params(vit_lp_model)

print("\nRN50 Size Info:")
rn50_params, rn50_train_params, rn50_size_mb = utils.calculate_model_size_params(rn50_lp_model)

## Linear Probing Training with Weights & Biases

We implement the training loop running the linear layer optimization for different $k$-shot values. The model uses Cross Entropy on similarity scores to identify the correct prompt.

In [ ]:
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm
import time

def train_linear_probe(model, model_name, preprocess, k, num_epochs=10):
    image_dir = EXTRACT_DIR if 'EXTRACT_DIR' in locals() else '/content/coco_subset_images'
    k_train_loader, k_test_loader = data_loader.create_dataloaders(
        train_max_data, eval_data, image_dir,
        preprocess, batch_size=config['training'].get('batch_size', 32), k=k
    )
    
    # Re-initialize the linear head for fresh training
    model.linear_head = nn.Linear(1, 1).to(device)
    optimizer = optim.Adam(model.linear_head.parameters(), lr=config['training'].get('learning_rate', 0.001))
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(k_train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch in pbar:
            images, text_candidates, targets, _, _ = batch
            images = images.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            logits = model(images, text_candidates)
            loss = criterion(logits, targets)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
            
            pbar.set_postfix({'loss': loss.item()})
            
        train_acc = correct / total
        
        # Evaluate
        eval_acc, eval_f1 = utils.evaluate_model(model, k_test_loader, device)
        
        print(f"Epoch {epoch+1} - Loss: {total_loss/len(k_train_loader):.4f} - Train Acc: {train_acc:.4f} - Eval Acc: {eval_acc:.4f} - Eval F1: {eval_f1:.4f}")
        
        wandb.log({
            f"{model_name}_{k}shot_train_loss": total_loss/len(k_train_loader),
            f"{model_name}_{k}shot_train_acc": train_acc,
            f"{model_name}_{k}shot_eval_acc": eval_acc,
            f"{model_name}_{k}shot_eval_f1": eval_f1
        })
        
        if eval_acc > best_acc:
            best_acc = eval_acc
            best_model_state = model.linear_head.state_dict()
            
    # Load best and save to file/artifact
    model.linear_head.load_state_dict(best_model_state)
    
    if not os.path.exists('../models'):
        os.makedirs('../models')
    save_path = f"../models/best_{model_name}_{k}shot.pth"
    torch.save(best_model_state, save_path)
    
    artifact = wandb.Artifact(f"{model_name}_{k}shot_model", type="model")
    artifact.add_file(save_path)
    wandb.log_artifact(artifact)
    
    return best_acc, eval_f1

k_shots_list = config['training']['k_shots']

if TRAIN_MODEL and subset_data:
    wandb.login()
    wandb.init(project=config['wandb']['project'], entity=config['wandb'].get('entity', None))
    
    for k in k_shots_list:
        print(f"\n--- Training ViT-B/32 Linear Probe for k={k} ---")
        train_linear_probe(vit_lp_model, "vit_b32", preprocess_vit, k, num_epochs=config['training'].get('epochs', 20))
        
        print(f"\n--- Training RN50 Linear Probe for k={k} ---")
        train_linear_probe(rn50_lp_model, "rn50", preprocess_rn50, k, num_epochs=config['training'].get('epochs', 20))
        
    wandb.finish()
else:
    print("Training is disabled (TRAIN_MODEL=False) or no data loaded. Skipping fine-tuning.")

## Model Evaluation and Metrics Comparison

Evaluate Zero-shot models to compare with Few-shot. Calculate Accuracy, F1 score and Time.

In [ ]:
import time

def evaluate_zero_shot(model, preprocess):
    if not subset_data: return 0, 0, 0
    image_dir = EXTRACT_DIR if 'EXTRACT_DIR' in locals() else '/content/coco_subset_images'
    _, test_loader = data_loader.create_dataloaders(
        train_max_data, eval_data, image_dir,
        preprocess, batch_size=config['training'].get('batch_size', 32), k=8
    )
    
    start_time = time.time()
    acc, f1 = utils.evaluate_model(model, test_loader, device)
    end_time = time.time()
    
    inference_time = end_time - start_time
    print(f"Zero-Shot Accuracy: {acc:.4f}")
    print(f"Zero-Shot F1 Score: {f1:.4f}")
    print(f"Inference Time for {len(eval_data)} samples: {inference_time:.4f}s")
    
    return acc, f1, inference_time

print("\n--- ViT-B/32 Zero-Shot Evaluation ---")
vit_zs_acc, vit_zs_f1, vit_zs_time = evaluate_zero_shot(vit_zs_model, preprocess_vit)

print("\n--- RN50 Zero-Shot Evaluation ---")
rn50_zs_acc, rn50_zs_f1, rn50_zs_time = evaluate_zero_shot(rn50_zs_model, preprocess_rn50)

## Prediction Visualizations and Interpretability

Interpret prediction results using Attention Rollout (Text encoder), GradCam (ResNet Image encoder) and EigenCam (ViT Image encoder) for correct and error classification cases.

In [ ]:
# (Placeholder for PyTorch GradCam/EigenCam visualization methods)
# In practice we would integrate a library like `pytorch-grad-cam` to show map overlays.
# Example: https://github.com/jacobgil/pytorch-grad-cam

print("Visualizations placeholder (Execute EigenCAM / GradCAM here)")
if subset_data:
    print("Simulating Visualizations for 5 images...")
    # Visualize top 5 correct and bottom 5 incorrect predictions
    pass